# 9-dars: Transfer learning — tayyor «ko’z»ni o’z rasmlaringizga o’rgatamiz

**Hikoya.** Tasavvur qiling: bir shifokor **10 yil** odamning tanasini o’rgangan. Endi u **yangi ixtisos** oladi: teri kasalliklari. Uni **noldan** o’qitmaymiz: u allaqachon **ko’rishni, tuzilishni, farqlashni** biladi. Unga faqat **bir necha yuz yangi rasm** ko’rsatish yetadi.

8-darsda CNN ni **noldan** o’qitdik: 50 000 rasm, bir necha daqiqa, natija **~58%**. Bugun esa **millionlab rasmda o’qitilgan tayyor tarmoqni** olamiz va uni **o’zimizning rasmlarimizga** moslaymiz. Bu **transfer learning** (bilimni ko’chirish). Natija: **bir necha yuz rasm va bir necha daqiqa** bilan noldan o’qitishdan **ancha yuqori aniqlik** (bizda 55.6% o’rniga 87.2%).

**Bugun siz:**
- rasmlar **papkalarda** turishini va **papka nomi = klass** ekanini ko’rasiz;
- tayyor **ResNet18** nimani bilishini tekshirasiz;
- uning **oxirgi qatlamini** almashtirib, o’zingizning klasslaringizga moslaysiz;
- **noldan** o’qitilgan CNN bilan **solishtirasiz**;
- **ma’lumot sifati** (kam rasm, noto’g’ri etiketka) natijaga qanchalik ta’sir qilishini **o’lchaysiz**;
- **o’z datasetingizni** yig’ishga tayyorlanasiz: 1-oy loyihasi **«Ko’z»**.

**Asosiy fikr:** katta tarmoqning **ichki qatlamlari** (chiziq, shakl, tuzilish) har qanday rasm uchun **foydali**. Faqat **oxirgi qaror** qatlamini o’zimiznikiga almashtiramiz.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**). Eng yaxshisi: **Runtime → Change runtime type → T4 GPU**.

## Bugungi so’zlar (shu jadvalga qaytib turing)

| So’z | Oddiy tilda | Misol |
|---|---|---|
| **transfer learning** | tayyor tarmoq bilimini yangi masalaga ko’chirish | shifokor yangi ixtisos oladi |
| **oldindan o’qitilgan model** | millionlab rasmda o’qitilgan tayyor tarmoq | ResNet18 |
| **ImageNet** | 1,2 million rasm, 1000 xil narsa: tayyor modellar shunda o’qigan | mushuk, mashina, banan… |
| **muzlatish** (freeze) | qatlam sonlarini o’zgartirmaslik | shifokorning eski bilimi |
| **fine-tuning** | oxirgi qatlamni almashtirib, yangi ma’lumotda o’qitish | yangi ixtisos |
| **dataset** | rasmlar to’plami, papkalarda klass bo’yicha | `olma/`, `banan/`… |
| **train / val** | o’rganish uchun / tekshirish uchun rasmlar | mashq / imtihon |
| **etiketka** | rasmning javobi (qaysi klass) | papka nomi |

## 0. Kutubxonalar

8-darsdagi kutubxonalar. Yangi: `models` (tayyor tarmoqlar), `zipfile` va `urllib` (fayl yuklash va ochish).

In [ ]:
import time
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

qurilma = "cuda" if torch.cuda.is_available() else "cpu"      # 7-dars: GPU bormi?
print("Qurilma:", qurilma)

## 1. Dataset: papka nomi = klass

**Hayotda:** telefon galereyasida albomlar bor: «Oila», «Sayohat», «Taomlar». Albom nomi — rasmning **belgisi**. Dataset ham shunday: **har klass uchun alohida papka**.

Biz **5 xil meva** rasmlari bilan boshlaymiz: olma, banan, apelsin, qulupnay, tarvuz. Bu ochiq dataset (**Snacks**, CC-BY 4.0 litsenziya). Yuklab, kerakli 5 ta klassni ochamiz (bir necha daqiqa kerak bo’lishi mumkin: fayl ~110 MB):

In [ ]:
MANBA = "https://huggingface.co/datasets/Matthijs/snacks/resolve/main/images.zip"
zip_fayl = Path("snacks.zip")

if not zip_fayl.exists():
    t0 = time.time()
    urllib.request.urlretrieve(MANBA, zip_fayl)
    print(f"yuklandi: {zip_fayl.stat().st_size / 1e6:.0f} MB, {time.time() - t0:.0f} soniya")
else:
    print("fayl allaqachon bor")

Zip ichida **papkalar** (`data/train/apple/…`) bor. Biz 5 ta klassni ochamiz va papkalarga **o’zbekcha nom** beramiz:

In [ ]:
nomlar = {"apple": "olma", "banana": "banan", "orange": "apelsin", "strawberry": "qulupnay", "watermelon": "tarvuz"}
bolimlar = {"train": "train", "val": "val", "test": "test"}

with zipfile.ZipFile(zip_fayl) as z:
    for fayl in z.namelist():
        q = fayl.split("/")                       # masalan: data / train / apple / abc.jpg
        if len(q) == 4 and q[0] == "data" and q[2] in nomlar and q[3].endswith(".jpg"):
            yol = Path("rasmlar") / q[1] / nomlar[q[2]] / q[3]
            yol.parent.mkdir(parents=True, exist_ok=True)
            yol.write_bytes(z.read(fayl))

for bolim in ["train", "val", "test"]:
    print(bolim, {p.name: len(list(p.glob("*.jpg"))) for p in sorted(Path("rasmlar", bolim).iterdir())})

Har bo’limda 5 ta papka. **train** — o’rganish uchun (~250 tadan), **val** (validation) — o’qitish paytida tekshirish uchun, **test** — oxirida **yakuniy imtihon** uchun (50 tadan). Tarmoq **val** va **test** rasmlarini **o’rganmaydi**.

Bu uchlik 4-darsdagi **train/test** g’oyasining davomi: **o’rganmagan** rasmlarda sinab ko’ramiz.

`ImageFolder` papkalar tuzilishini **o’zi** o’qiydi: papka nomi — klass:

In [ ]:
sinflar = sorted(p.name for p in Path("rasmlar/train").iterdir())
print("klasslar:", sinflar)

xom = datasets.ImageFolder("rasmlar/train")            # hali o'zgartirishsiz, asl rasmlar
print("train rasmlari:", len(xom))
print("birinchi rasm:", xom.samples[0])                 # (fayl yo'li, klass raqami)

`(fayl yo’li, klass raqami)`: **klass raqami** papkaning alifbo tartibidagi o’rni (apelsin = 0, banan = 1…). Ketma-ket 10 ta tasodifiy rasmni ko’ramiz:

In [ ]:
import random
random.seed(0)
tanlangan = random.sample(range(len(xom)), 10)

plt.figure(figsize=(14, 3))
for k, i in enumerate(tanlangan):
    rasm, javob = xom[i]
    plt.subplot(2, 5, k + 1)
    plt.imshow(rasm); plt.title(sinflar[javob], fontsize=10); plt.axis("off")
plt.tight_layout()
plt.show()

Rasmlar **turli o’lchamda**, turli burchakdan, turli fonda: do’kon peshtaxtasi, daraxtdagi olma, taqsimchadagi tarvuz. Mana **haqiqiy** rasmlar. 8-darsdagi 32 × 32 rasmdan ancha yaxshi.

**Mustaqil ish 5** (dars oxirida): o’z datasetingizni **tekshirish** funksiyasi. Avval asosiy ishni qilamiz.

## 2. Rasmni tarmoqqa tayyorlash

**Hayotda:** har xil o’lchamdagi rasmlarni **bitta albom formatiga** keltirasiz. Tarmoq ham **bir xil o’lchamli** rasmlarni kutadi.

Tayyor tarmoq **224 × 224** rasmlarda o’qigan. Shuning uchun rasmni:
1. **224 × 224** ga keltiramiz (`Resize`),
2. **tensorga** aylantiramiz (`ToTensor`, 0..1),
3. **ImageNet o’rtachasi va tarqalishi** bo’yicha **tekislaymiz** (`Normalize`): tayyor tarmoq shu qoida bilan o’qigan, biz ham xuddi shunday «ko’zoynak» taqamiz.

In [ ]:
mean = [0.485, 0.456, 0.406]          # ImageNet ning o'rtacha rangi (qizil, yashil, ko'k)
std = [0.229, 0.224, 0.225]           # va tarqalishi

tayyorla = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_data = datasets.ImageFolder("rasmlar/train", transform=tayyorla)
val_data = datasets.ImageFolder("rasmlar/val", transform=tayyorla)
test_data = datasets.ImageFolder("rasmlar/test", transform=tayyorla)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=50)
test_loader = DataLoader(test_data, batch_size=50)

rasm, javob = train_data[0]
print("bitta rasm:", tuple(rasm.shape), "klass:", sinflar[javob])
print("train:", len(train_data), " val:", len(val_data), " test:", len(test_data))

Bitta rasm **`[3, 224, 224]`**: 3 kanal, 224 × 224. Bu 8-darsdagi `[3, 32, 32]` dan **49 marta katta** (224² / 32² = 49): ancha ko’proq tafsilot.

## 3. Boshlang’ich nuqta: CNN ni noldan o’qitamiz (8-dars)

**Hayotda:** yangi usulning foydasini bilish uchun avval **eski usul** qanday ishlashini o’lchaysiz. Shuning uchun avval 8-darsdagi kabi CNN ni **noldan** o’qitamiz va natijani yozib qo’yamiz.

Rasmlarni **64 × 64** ga kichraytiramiz (noldan o’qitish tez bo’lsin), CNN ni 8-darsdagi kabi yig’amiz (faqat 3-blok qo’shilgan):

In [ ]:
kichik_tf = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor()])
kichik_train = DataLoader(datasets.ImageFolder("rasmlar/train", transform=kichik_tf), batch_size=32, shuffle=True)
kichik_val = DataLoader(datasets.ImageFolder("rasmlar/val", transform=kichik_tf), batch_size=50)


def cnn_yasa(klass_soni):
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 64 -> 32
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 32 -> 16
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),      # 16 -> 8
        nn.Flatten(), nn.Linear(64 * 8 * 8, klass_soni),
    )


print("CNN dagi sonlar:", sum(p.numel() for p in cnn_yasa(5).parameters()))

O’qitish tsikli — 7- va 8-darsdagi **bir xil** tsikl. Faqat bitta o’zgarish: tekshirish **val** rasmlarida. Tayyor, ko’chirib qo’ydik:

In [ ]:
def aniqlik(model, yuklagich):
    model.eval()
    togri, jami = 0, 0
    with torch.no_grad():
        for x, y in yuklagich:
            x, y = x.to(qurilma), y.to(qurilma)
            togri += (model(x).argmax(dim=1) == y).sum().item()
            jami += len(y)
    return togri / jami


def o_qit(model, yuklagich, tekshir, epochlar=3, lr=0.001, muzlatilgan=False):
    model = model.to(qurilma)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)   # faqat o'rganadigan sonlar
    natijalar = []
    for epoxa in range(epochlar):
        model.eval() if muzlatilgan else model.train()       # muzlatilgan qatlamlar bir xil ishlashi uchun
        jami_loss = 0.0
        for x, y in yuklagich:
            x, y = x.to(qurilma), y.to(qurilma)
            loss = loss_fn(model(x), y)            # 1. oldinga, 2. loss
            optimizer.zero_grad()                  # 3. eski ta'sirlarni o'chir
            loss.backward()                        # 4. orqaga
            optimizer.step()                       # 5. qadam
            jami_loss += loss.item()
        aniq = aniqlik(model, tekshir)
        natijalar.append((jami_loss / len(yuklagich), aniq))
        print(f"{epoxa + 1}-epoxa: loss = {natijalar[-1][0]:.4f}, val aniqligi = {aniq * 100:.2f}%")
    return natijalar

Noldan o’qitamiz (5 epoxa):

In [ ]:
torch.manual_seed(1)
t0 = time.time()
noldan_natija = o_qit(cnn_yasa(5), kichik_train, kichik_val, epochlar=5)
noldan_vaqt = time.time() - t0
print(f"vaqt: {noldan_vaqt:.0f} soniya")

Taxminan **55.6%**. Tasodif **20%** (5 ta klass). CNN **o’rgandi**, lekin sust. Sabab: **atigi 1248 rasm** va noldan o’qitish. 8-darsda 50 000 rasm bilan ham 58% edi. Endi **tayyor tarmoq** bilan sinab ko’ramiz.

## 4. Tayyor tarmoq: ResNet18

**Hayotda:** o’zingiz mashina yasamasdan, **tayyor mashinani** sotib olasiz va **rangini** o’zgartirasiz. ResNet18 — **tayyor mashina**: uni Microsoft tadqiqotchilari **1,2 million rasmda** (ImageNet, 1000 xil narsa) o’qitgan.

PyTorch uni bir qatorda yuklaydi (birinchi marta ~45 MB yuklanadi):

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("sonlar soni:", f"{sum(p.numel() for p in resnet.parameters()):,}")
print("oxirgi qatlam:", resnet.fc)

- **11 689 512** ta son: bizning CNN dan **~265 marta** katta. Bu sonlar **allaqachon o’rganilgan**;
- **oxirgi qatlam** `fc`: `Linear(512 → 1000)`. Oldin 512 ta son kiradi (rasmning «xulosasi»), chiqishda **1000 ta** ball: ImageNet ning 1000 ta narsasi.

**Tayyor tarmoq rasmda nima ko’radi?** Bizning banan rasmini **hech qanday o’qitishsiz** unga beramiz va u **o’zining 1000 ta** javobidan qaysini tanlashini ko’ramiz:

In [ ]:
imagenet_nomlar = models.ResNet18_Weights.IMAGENET1K_V1.meta["categories"]       # 1000 ta nom (inglizcha)

resnet.eval()
banan_fayl = sorted(Path("rasmlar/val/banan").glob("*.jpg"))[0]
from PIL import Image
banan_rasm = Image.open(banan_fayl).convert("RGB")

with torch.no_grad():
    ballar = resnet(tayyorla(banan_rasm).unsqueeze(0))[0]
    ehtimol = torch.softmax(ballar, dim=0)
    eng_yaxshi = ehtimol.topk(3)

plt.figure(figsize=(3, 3)); plt.imshow(banan_rasm); plt.axis("off"); plt.show()
for p, i in zip(eng_yaxshi.values, eng_yaxshi.indices):
    print(f"{imagenet_nomlar[i]:<20} {p.item() * 100:5.1f}%")

Tayyor tarmoq **o’qitilmagan holda** banan rasmini «**banana**» deb taniydi. U ImageNet da banan ko’p ko’rgan. Lekin **bizning 5 klassimizni bilmaydi**: u 1000 ta javobdan tanlaydi (ichida «tarvuz» yo’q!). Shuning uchun **javob qatlamini** almashtiramiz, **ichki qatlamlar** esa o’zgarmaydi.

## 5. Oxirgi qatlamni almashtiramiz

Bu transfer learning ning **asosiy qadami**. Tarmoq **ikki qismdan** iborat:

| Qism | Nima qiladi | Nima qilamiz |
|---|---|---|
| **ichki qatlamlar** (chiziq → shakl → tuzilish) | rasmdan **512 ta son** «xulosa» chiqaradi | **muzlatamiz**: o’zgartirmaymiz |
| **oxirgi qatlam** `fc` | 512 ta sondan **javob** chiqaradi | **almashtiramiz**: 1000 → 5, tasodifiy sonlar bilan, o’qitamiz |

**Hayotda:** shifokorning **umumiy bilimi** (ichki qatlamlar) qoladi, yangi **ixtisos** (oxirgi qatlam) o’rganiladi.

**Mustaqil ish 2.** `model_yasa(klass_soni)` funksiyasini yozing: tayyor ResNet18 oling, **hamma** sonlarni muzlating (`requires_grad = False`), oxirgi qatlamni yangi `nn.Linear(512, klass_soni)` ga almashtiring (yangi qatlam **o’zi** o’rganadigan bo’ladi).

In [ ]:
# TODO: model_yasa funksiyasini to'ldiring
def model_yasa(klass_soni):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = True              # TODO: False yozing (hamma sonlarni muzlatamiz)
    # TODO: oxirgi qatlamni almashtiring: model.fc = nn.Linear(512, klass_soni)
    return model


model = model_yasa(len(sinflar))
o_rganadigan = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert o_rganadigan == 512 * 5 + 5, f"o'rganadigan sonlar {o_rganadigan} ta, lekin 2565 bo'lishi kerak. Hamma sonlarni muzlatdingizmi (requires_grad = False) va model.fc = nn.Linear(512, klass_soni) yozdingizmi?"
print("o'rganadigan sonlar:", o_rganadigan)

**2565 ta son** (512 · 5 + 5) o’rganadi, qolgan **11 689 512** ta son **muzlatilgan**. Shu kichik qatlamga **hamma o’rganish** yuklanadi. Shuning uchun o’qitish **tez** va **kam rasm** yetadi.

Yangi qatlam qanday ko’rinadi va chiqishi nechta?

In [ ]:
print(model.fc)
x = rasm.unsqueeze(0)
print("kirish:", tuple(x.shape), "-> chiqish:", tuple(model(x).shape))

Chiqish **`[1, 5]`**: 5 ta klass uchun 5 ta ball (7-dars: eng balandi — javob).

## 6. O’qitamiz

**Hayotda:** shifokor yangi ixtisos bo’yicha **bir necha hafta** o’qiydi, **10 yil** emas. Biz ham **3 epoxa**.

`muzlatilgan=True` — ichki qatlamlar **o’zgarmasligi** uchun (7–8-darsdagi tsikl bilan **bir xil**, qolgan hammasi o’sha):

In [ ]:
torch.manual_seed(1)
t0 = time.time()
transfer_natija = o_qit(model, train_loader, val_loader, epochlar=3, muzlatilgan=True)
transfer_vaqt = time.time() - t0
print(f"vaqt: {transfer_vaqt:.0f} soniya")

Taxminan **87.2%** (val). Noldan **55.6%** edi. **Qanchalik farq!** Bizda **bor-yo’g’i 1248 ta rasm**, 3 epoxa va **2565 ta o’rganadigan son**. Qolgan hammasini tayyor tarmoq **o’zi** bilar edi.

Bu **dars natijasi**: o’z ma’lumotimizda o’qitilgan **1-versiya model**. **Yakuniy imtihon** (test): tarmoq ko’rmagan rasmlar.

In [ ]:
test_aniq = aniqlik(model, test_loader)
print(f"test aniqligi: {test_aniq * 100:.1f}%  ({len(test_data)} ta rasm)")

**Nega val bilan emas, test bilan?** Biz o’qitish paytida **val** natijasiga qarab sozlaymiz (epoxa soni, lr). Shuning uchun val biroz «ko’rilgan» bo’lib qoladi. **Test** esa **faqat bir marta, oxirida** ishlatiladi: halol baho.

Taqqoslash grafigi:

In [ ]:
noldan_aniq = noldan_natija[-1][1] * 100
transfer_aniq = transfer_natija[-1][1] * 100

plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
plt.bar(["noldan CNN", "transfer"], [noldan_aniq, transfer_aniq], color=["#94a3b8", "#00D1D1"])
plt.ylabel("val aniqligi, %"); plt.title("Aniqlik")
for i, v in enumerate([noldan_aniq, transfer_aniq]):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center")

plt.subplot(1, 2, 2)
plt.bar(["noldan CNN", "transfer"], [noldan_vaqt, transfer_vaqt], color=["#94a3b8", "#00D1D1"])
plt.ylabel("soniya"); plt.title("O'qitish vaqti")
plt.tight_layout()
plt.show()

Chapda **aniqlik**: transfer ancha baland. O’ngda **vaqt**: CPU da transfer sekinroq bo’lishi mumkin (224 × 224 va katta tarmoq: har rasm **49 marta** ko’p son), lekin **GPU** da tezlik farqi yo’qoladi. Eng muhimi: **natija** va **kam rasm**.

**Sinab ko’ring.** Tayyor tarmoq **bizning 5 klassga kirmaydigan** rasmni ko’rsatsak, nima deydi? Masalan, **tort** rasmini beramiz (hech qaysi meva emas). Avval taxmin qiling: «bilmayman» deydimi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Tarmoq **«bilmayman»** demaydi: u faqat **5 ta javobdan** tanlaydi va tortni ham **qaysidir meva** deydi. Bu muhim saboq: **model faqat o’rgangan klasslarni biladi**. 10-darsda **ishonch** va chegaralar haqida gaplashamiz.

## 7. Model qayerda adashadi?

**Hayotda:** imtihondan keyin **xatolarni** ko’rasiz: qaysi savolda adashdim? Model ham: qaysi rasmlarda adashgan?

In [ ]:
model.eval()
xato_rasmlar = []                          # (test raqami, haqiqiy klass, model javobi, ishonch)
with torch.no_grad():
    for k, (x, y) in enumerate(test_loader):
        ehtimol = torch.softmax(model(x.to(qurilma)), dim=1).cpu()
        bashorat = ehtimol.argmax(dim=1)
        for j in range(len(y)):
            if bashorat[j] != y[j]:
                xato_rasmlar.append((k * 50 + j, y[j].item(), bashorat[j].item(), ehtimol[j].max().item()))

print("xato qilgan rasmlar:", len(xato_rasmlar), "ta, jami", len(test_data), "tadan")
for k, nom in enumerate(sinflar):
    print(f"{nom:<10} xato: {sum(1 for x in xato_rasmlar if x[1] == k)} ta")

Beshta tasodifiy xatoni ko’ramiz (**haqiqiy → model javobi**, ishonchi bilan). Siz uni qayta ishga tushirsangiz, boshqa rasmlar chiqadi:

In [ ]:
import random
fayllar = [p for p, _ in test_data.samples]

plt.figure(figsize=(14, 3.4))
random.seed(1)
for k, (i, haqiqiy, taxmin, ishonch) in enumerate(random.sample(xato_rasmlar, 5)):
    plt.subplot(1, 5, k + 1)
    plt.imshow(Image.open(fayllar[i]).convert("RGB"))
    plt.title(f"{sinflar[haqiqiy]} -> {sinflar[taxmin]}\n({ishonch * 100:.0f}%)", fontsize=10)
    plt.axis("off")
plt.show()

Bizning ishga tushirishda xatolar ko’pincha shunday rasmlarda: **o’xshash rang** (sariq olma → apelsin), **bo’laklangan yoki quritilgan** meva (bo’laklangan banan, quritilgan apelsin), **boshqa narsalar bilan birga** (limon yonidagi banan, tarvuz ko’targan odam). Ba’zi rasmlarda **etiketkaning o’zi xato** bo’lishi mumkin. **Datasetni ham tekshirish kerak!** 10-darsda **confusion matrix** va xatolar tahlili bilan bu ishni chuqurroq qilamiz.

## 8. Ma’lumot sifati: nega bu muhim?

**Hayotda:** o’quvchiga **kam** yoki **noto’g’ri** kitob bersangiz, u yaxshi o’qiy olmaydi: **ma’lumot — tarmoq uchun kitob**. Endi buni **o’lchaymiz**.

Ikkita tajriba: (1) **rasmlar soni** natijaga qanday ta’sir qiladi, (2) **noto’g’ri etiketka** nima qiladi.

**Tezlashtirish.** Ichki qatlamlar **muzlatilgan**, shuning uchun har rasm uchun ularning **512 ta son chiqishini** bir marta hisoblab **saqlab** qo’yamiz. Keyin faqat **oxirgi qatlamni** o’qitamiz: bu **soniyalar** oladi, tajribalarni **ko’p marta** takrorlash mumkin. Kod tayyor:

In [ ]:
def xususiyat_ol(yuklagich):
    asos = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    asos.fc = nn.Identity()                       # oxirgi qatlamsiz: faqat 512 ta son chiqadi
    asos = asos.to(qurilma).eval()
    X, Y = [], []
    with torch.no_grad():
        for x, y in yuklagich:
            X.append(asos(x.to(qurilma)).cpu())
            Y.append(y)
    return torch.cat(X), torch.cat(Y)


t0 = time.time()
Xtr, ytr = xususiyat_ol(DataLoader(train_data, batch_size=50))
Xva, yva = xususiyat_ol(val_loader)
print(f"hisoblandi: {time.time() - t0:.0f} soniya. Xtr: {tuple(Xtr.shape)}, ytr: {tuple(ytr.shape)}")

Har rasm **512 ta son** bilan ifodalandi («rasmning xulosasi»): 1248 × 512. Endi **oxirgi qatlam** o’qitish funksiyasi (mini-o’qitish):

In [ ]:
def bosh_oqit(X, y, Xv, yv, epochlar=40, lr=0.01, seed=1):
    torch.manual_seed(seed)
    bosh = nn.Linear(512, len(sinflar))
    optimizer = torch.optim.Adam(bosh.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochlar):
        for i in torch.randperm(len(X)).split(64):          # 64 tadan bo'lak (batch)
            loss = loss_fn(bosh(X[i]), y[i])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    with torch.no_grad():
        return (bosh(Xv).argmax(dim=1) == yv).float().mean().item()


print(f"hamma {len(Xtr)} ta rasm: val aniqligi = {bosh_oqit(Xtr, ytr, Xva, yva) * 100:.1f}%")

Natija 3 epoxali o’qitish bilan bir xil (87.2%): demak, tezlashtirish **to’g’ri** ishlaydi.

### 8.1. Kam rasm bo’lsa-chi?

**Mustaqil ish 3.** `har_klassdan(y, n)` funksiyasini yozing: **har klassdan faqat `n` ta** rasmning raqamini qaytaring (hamma klassdan **teng**).

In [ ]:
# TODO: har_klassdan funksiyasini to'ldiring
def har_klassdan(y, n):
    tanlangan = []
    for k in range(len(sinflar)):
        indekslar = (y == k).nonzero().flatten()     # shu klassdagi hamma rasmlarning raqamlari
        # TODO: shu klassdan birinchi n tasini tanlangan ga qo'shing: tanlangan.extend(indekslar[:n].tolist())
    return tanlangan


t = har_klassdan(ytr, 10)
assert len(t) == 50, f"{len(t)} ta rasm qaytdi, lekin 5 klass x 10 = 50 bo'lishi kerak. tanlangan.extend(indekslar[:n].tolist()) yozdingizmi?"
print("tanlandi:", len(t), "ta rasm")

Endi **har klassdan** 2, 5, 10, 25, 50, 100 va 250 ta rasm bilan o’qitib, **val** aniqligini ko’ramiz:

In [ ]:
sonlar = [2, 5, 10, 25, 50, 100, 250]
aniqliklar = []
for n in sonlar:
    t = har_klassdan(ytr, n)
    a = bosh_oqit(Xtr[t], ytr[t], Xva, yva) * 100
    aniqliklar.append(a)
    print(f"har klassdan {n:>3} ta ({len(t):>4} rasm): val aniqligi = {a:.1f}%")

plt.figure(figsize=(6, 3.4))
plt.plot(sonlar, aniqliklar, "o-", color="#00D1D1")
plt.xscale("log"); plt.xlabel("har klassdan rasm soni"); plt.ylabel("val aniqligi, %"); plt.grid(alpha=0.3)
plt.show()

**Natija:** har klassdan atigi **10 ta** (jami 50) rasm bilan tayyor tarmoq **62.4%** berdi. Noldan o’qitilgan CNN esa **1248** rasmda 55.6% edi. **Lekin** rasm sonini oshirish baribir kerak: 10 tadan 100 taga aniqlik **62.4%** dan **86.0%** ga ko’tarildi. 100 dan 250 gacha o’sish sekinlashdi (86.0% → 87.2%). Shuning uchun reja: **har klassga 100+ rasm** (uy ishida 150+).

(Eslatma: val bor-yo’g’i 250 ta rasm, shuning uchun bir necha foizlik tasodifiy farq normal.)

### 8.2. Noto’g’ri etiketka

**Hayotda:** o’quvchining daftarida **javoblar xato** yozilgan bo’lsa, u **xatoni** o’rganadi. Datasetda ham shunday: kimdir **banan** rasmini **olma** papkasiga solib qo’ygan bo’lishi mumkin. Buni **o’lchaymiz**: etiketkalarning **bir qismini ataylab buzamiz**.

**Mustaqil ish 4.** `etiketka_buz(y, ulush)` funksiyasini yozing: etiketkalarning `ulush` qismini **boshqa** klassga almashtiring. Tayyor qismlar: `buzilgan` — buziladigan rasmlar raqami, `siljish` — har biriga 1 dan 4 gacha tasodifiy son (shuni qo’shib, 5 ga bo’lgandagi **qoldiq** olinsa, **albatta boshqa** klass chiqadi).

In [ ]:
# TODO: etiketka_buz funksiyasini to'ldiring
def etiketka_buz(y, ulush, seed=0):
    y = y.clone()
    gen = torch.Generator().manual_seed(seed)
    buzilgan = torch.randperm(len(y), generator=gen)[:int(len(y) * ulush)]            # buziladigan rasmlar
    siljish = torch.randint(1, len(sinflar), (len(buzilgan),), generator=gen)            # 1..4
    # TODO: y[buzilgan] = (y[buzilgan] + siljish) % len(sinflar) yozing
    return y


y_buzuq = etiketka_buz(ytr, 0.3)
farq = (y_buzuq != ytr).float().mean().item()
assert abs(farq - 0.3) < 0.01, f"etiketkalarning {farq * 100:.0f}% i o'zgardi, lekin 30% bo'lishi kerak. y[buzilgan] = (y[buzilgan] + siljish) % len(sinflar) yozdingizmi?"
print(f"etiketkalarning {farq * 100:.0f}% i buzildi")

Endi **0%, 10%, 20%, 30%, 50%** etiketka buzilganda natijani o’lchaymiz. **Muhim:** buzilgan **faqat train** etiketkalari; **val** (tekshirish) to’g’ri qoladi.

In [ ]:
ulushlar = [0, 0.1, 0.2, 0.3, 0.5]
buzuq_aniq = []
for u in ulushlar:
    a = bosh_oqit(Xtr, etiketka_buz(ytr, u), Xva, yva) * 100
    buzuq_aniq.append(a)
    print(f"etiketkalarning {u * 100:>3.0f}% i xato: val aniqligi = {a:.1f}%")

plt.figure(figsize=(6, 3.4))
plt.plot([u * 100 for u in ulushlar], buzuq_aniq, "o-", color="#ef4444")
plt.xlabel("xato etiketka, %"); plt.ylabel("val aniqligi, %"); plt.grid(alpha=0.3)
plt.show()

**Xulosa:** etiketkalarning atigi **10%** i xato bo’lganda ham aniqlik 87.2% dan **75.6%** ga tushdi; 30% da **60.4%**, 50% da **43.2%**. Tayyor tarmoq xato javobni ham **o’rganib oladi**. Datasetni **yig’ishda ehtiyot bo’ling**: har rasm to’g’ri papkada ekanini tekshiring. Bitta xato papka — bir necha o’nlab xato etiketka.

**Sinab ko’ring.** Yana bir xato turi: **bitta klass** juda **kam**, boshqalari ko’p (muvozanatsiz dataset). Masalan, olma 250 ta, tarvuz faqat 10 ta. Avval taxmin qiling: model qaysi klassda yomon ishlaydi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 9. O’z datasetingiz: «Ko’z» loyihasi

**Hayotda:** siz **o’zingiz biladigan** va **sizga qiziq** narsa ustida ishlaysiz: oshxonadagi taomlar, ko’chadagi mashinalar, bog’dagi o’simliklar. Shunda natija **sizniki**.

**«Ko’z»** — 1-oyning loyihasi: **o’zingiz tanlagan mavzuda** rasm tanuvchi model, keyin uni **internetga** chiqaramiz (11-dars).

**Mavzu tanlang** (3–6 ta klass bo’lsin):
- **o’zbek taomlari** (plov, somsa, manti, lag’mon…),
- **mevalar va sabzavotlar**,
- **avtomobil markalari**,
- **o’simlik kasalliklari** (bargdagi dog’lar),
- yoki o’zingizniki (uy hayvonlari, kiyim, bino…).

**Rasm yig’ish qoidalari:**
1. Har klassga **kamida 100** ta rasm (uy ishida **150+**).
2. **Xilma-xil**: turli fon, yorug’lik, burchak, masofa. Hammasi bir xil joyda olingan bo’lsa, model **fonni** o’rganib qo’yadi.
3. **Muvozanat**: klasslar taxminan **teng** (60–150, 100–100…).
4. **Etiketka to’g’ri**: rasm to’g’ri papkada ekanini tekshiring.
5. **Insonlar yuzi** va **boshqalarning shaxsiy** rasmlarini **olmang** (ruxsatsiz). Internetdan olsangiz, **litsenziyasi** ochiq bo’lsin.
6. Takror rasmlarni olib tashlang.

**Papka tuzilishi** (huddi shunday bo’lishi shart):
```
mening_rasmlarim/
    plov/    rasm1.jpg  rasm2.jpg ...
    somsa/   ...
    manti/   ...
```
Keyin **train** va **val** ga bo’linadi (80% / 20%): bu ishni kod qiladi.

**Mustaqil ish 5.** Datasetni **tekshirish** funksiyasi. U papkani o’qiydi, **har klassda nechta** rasm borligini hisoblaydi va **ogohlantiradi**: biror klassda 100 dan kam bo’lsa yoki klasslar **juda notekis** bo’lsa (eng ko’pi eng kamidan 2 marta ko’p). Avval namuna dataset (bizning mevalar) da sinaymiz.

In [ ]:
# TODO: dataset_tekshir funksiyasini to'ldiring
def dataset_tekshir(papka, kamida=100):
    sonlar = {}
    for klass in sorted(Path(papka).iterdir()):
        if klass.is_dir():
            sonlar[klass.name] = 0          # TODO: len(list(klass.glob("*.jpg"))) yozing (shu klassdagi .jpg soni)
    ogohlantirishlar = []
    for nom, soni in sonlar.items():
        print(f"{nom:<12} {soni:>4} ta")
        if soni < kamida:
            ogohlantirishlar.append(f"{nom}: {soni} ta, kamida {kamida} kerak")
    if max(sonlar.values()) > 2 * min(sonlar.values()):
        ogohlantirishlar.append("klasslar notekis: eng ko'pi eng kamidan 2 martadan ko'p")
    return sonlar, ogohlantirishlar


sonlar_t, ogoh = dataset_tekshir("rasmlar/train")
assert sonlar_t["olma"] == 249 or sonlar_t["olma"] > 100, f"olma klassida {sonlar_t['olma']} ta. len(list(klass.glob('*.jpg'))) yozdingizmi?"
print("ogohlantirishlar:", ogoh if ogoh else "yo'q")

Bizning mevalar datasetida hamma klass **~250** ta: ogohlantirish yo’q. **Sizning** datasetingizda bo’lsa, **ko’proq rasm yig’ing** (uy ishi).

**O’z datasetingizni notebook’ga qanday olib kirasiz:**
1. Rasmlarni **papkalarga** soling (yuqoridagi tuzilishda), papkani **zip** qiling (`mening_rasmlarim.zip`).
2. Colab’da chap tomondagi **papka belgisi (📁)** → **yuklash (↑)** tugmasi bilan zip’ni yuklang (yoki Google Drive’dan oling).
3. Quyidagi funksiya zip’ni ochadi va **train / val** ga bo’ladi (80% / 20%). **Hozir ishga tushirish shart emas**: uy ishida kerak bo’ladi. Funksiyani **avval o’qib chiqing**, tayyor:

In [ ]:
import shutil
import random


def datasetni_bol(zip_yoli, chiqish="mening_data", val_ulushi=0.2, seed=0):
    ochilgan = Path(chiqish + "_xom")
    with zipfile.ZipFile(zip_yoli) as z:
        z.extractall(ochilgan)                               # zip ichidagi papkalarni ochdik
    asosiy = ochilgan
    ichkari = [p for p in ochilgan.iterdir() if p.is_dir()]
    if len(ichkari) == 1 and not any(p.suffix.lower() in (".jpg", ".jpeg", ".png") for p in ochilgan.iterdir()):
        asosiy = ichkari[0]                                  # zip bitta papka ichida bo'lsa, o'shanga kiramiz

    rnd = random.Random(seed)
    for klass in sorted(p for p in asosiy.iterdir() if p.is_dir()):
        rasmlar = sorted(p for p in klass.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
        rnd.shuffle(rasmlar)
        val_soni = int(len(rasmlar) * val_ulushi)
        for k, r in enumerate(rasmlar):
            bolim = "val" if k < val_soni else "train"
            maqsad = Path(chiqish) / bolim / klass.name
            maqsad.mkdir(parents=True, exist_ok=True)
            shutil.copy(r, maqsad / r.name)
    for bolim in ["train", "val"]:
        print(bolim, {p.name: len(list(p.iterdir())) for p in sorted((Path(chiqish) / bolim).iterdir())})

**Muhim:** `ImageFolder` `.jpg`, `.jpeg` va `.png` ni o’qiydi, lekin **`dataset_tekshir`** faqat `.jpg` ni sanaydi: o’z datasetingizda `.png` bo’lsa, funksiyani moslashtiring.

Keyin yuqoridagi **2–7-bo’limlarning** kodini o’z papkalaringizga ko’rsatasiz: `"rasmlar/train"` o’rniga `"mening_data/train"` va hokazo. Qolgan kod **o’zgarmaydi**: klasslar soni `len(sinflar)` dan o’zi aniqlanadi.

## 10. Siz buni har kuni ishlatasiz

Bugungi usul — **transfer learning** — zamonaviy sun’iy intellektning **asosiy** usuli:
- **Tibbiyot**: rentgen va teri kasalliklarini aniqlash; tayyor model yuz minglab rasmda sozlanadi;
- **Qishloq xo’jaligi**: barg kasalliklari (telefon rasmidan);
- **Chakana savdo**: javondagi mahsulotni avtomatik tanish;
- **Telefon galereyasi**: sizning rasmlaringizga moslashadi (yuzlar, joylar);
- **ChatGPT va boshqa katta modellar**: ular ham **oldindan o’qitilgan**, keyin sozlangan (2-oyda ko’ramiz).

Hammasida **bir xil g’oya**: katta modelni **noldan** o’qitmaymiz, **tayyorini** o’z masalamizga moslaymiz.

**Keyingi dars (10-dars): modelni baholash va yaxshilash.** Bugun model **adashgan** rasmlarni ko’rdik. Keyingi darsda **confusion matrix** (kim kimni adashtiradi), **Grad-CAM** (model rasmning **qayeriga** qaradi) va **«yolg’on o’rganish»** (model ob’ektni emas, **fonni** tanib qo’yishi mumkin) ni ko’ramiz.

## Uy ishi

**1. Datasetni yig’ing va kengaytiring.** «Ko’z» mavzusini tanlang (3–6 klass). **Har klassga 150+** rasm yig’ing: turli fon, yorug’lik, burchak. `dataset_tekshir` bilan tekshiring: ogohlantirish **bo’lmasligi** kerak.

**2. Datasetni tozalang.** Papkalarni **ko’zdan kechiring**: noto’g’ri papkadagi rasm, takror, juda xira yoki bo’sh rasmlarni **olib tashlang**. Nechta rasm olib tashlaganingizni yozing.

**3. 1-versiya modelni o’qiting.** `datasetni_bol` bilan bo’ling, `model_yasa(len(sinflar))` bilan o’qiting (3 epoxa). **Val aniqligini** yozing.

**4. (Qisqa izoh, 2–3 gap).** Qaysi klass eng ko’p adashadi? Nega, sizningcha? (Maslahat: 7-bo’limdagi xato rasmlarni ko’ring.)

Boshlash uchun shablon (o’z papkangizga yo’l qo’yasiz):

In [ ]:
# 1. Datasetni bo'ling (zip ni yuklab, nomini yozing):
# datasetni_bol("mening_rasmlarim.zip", chiqish="mening_data")

# 2. Tekshiring:
# sonlar, ogoh = dataset_tekshir("mening_data/train")
# print(ogoh)

# 3. O'qiting:
# mening_train = datasets.ImageFolder("mening_data/train", transform=tayyorla)
# mening_val = datasets.ImageFolder("mening_data/val", transform=tayyorla)
# sinflar = mening_train.classes
# torch.manual_seed(1)
# mening_model = model_yasa(len(sinflar))
# o_qit(mening_model, DataLoader(mening_train, batch_size=32, shuffle=True),
#       DataLoader(mening_val, batch_size=50), epochlar=3, muzlatilgan=True)

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy): ichki qatlamni ham o’qitish (fine-tuning)

Biz **hamma ichki qatlamni muzlatdik**. **Fine-tuning** degani: oxirgi qatlam **bilan birga ichki qatlamlarning oxirgisini** ham **bir oz** o’qitish.

1. `model_yasa` ni nusxalab, **`layer4`** (ResNet ning oxirgi bloki) sonlarini ham `requires_grad = True` qiling (`for p in model.layer4.parameters(): p.requires_grad = True`).
2. Kichikroq **o’rganish tezligi** bilan o’qiting (`lr=0.0001`): ichki qatlamlar **sekin** o’zgarsin.
3. Natijani oddiy transfer bilan solishtiring: aniqlik oshdimi? O’qitish vaqti-chi? (Maslahat: `muzlatilgan=False` bo’lishi kerak, nega?)

**Yana:** ResNet18 o’rniga **MobileNetV3** (telefon uchun yengil model: `models.mobilenet_v3_small`) ni sinab ko’ring. Uning oxirgi qatlami `model.classifier[3]` da.

## Yakun

Bugun siz:
- rasmlar **papkalarda** turishini va **papka nomi = klass** ekanini ko’rdingiz;
- tayyor **ResNet18** (millionlab rasmda o’qitilgan) ni oldingiz va uning **oxirgi qatlamini** almashtirdingiz;
- **noldan** o’qitilgan CNN bilan **solishtirdingiz** va farqni **o’lchadingiz**;
- **kam rasm** va **noto’g’ri etiketka** natijaga qanday ta’sir qilishini **o’lchadingiz**;
- o’z datasetingizni **tekshirishni** va «Ko’z» loyihangiz mavzusini tanlashni boshladingiz.

Keyingi darsda (10-dars): modelni **baholash va yaxshilash**: confusion matrix, Grad-CAM, «yolg’on o’rganish», model kartochkasi.